In [0]:
dbutils.widgets.text("order_id", "1")

In [0]:

from pyspark.sql import SparkSession

df = spark.read.option("header", "true").csv("/Volumes/workspace/test_schema/sales/sales*.csv")
display(df)

In [0]:

from pyspark.sql.functions import col

df_sales = df.withColumn(
    "sales_amount",
    col("quantity").cast("int") * col("price").cast("int")
)

display(df_sales)

In [0]:

df_sales.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/test_schema/sales/retail_sales_delta")

In [0]:
%sql

CREATE or replace TABLE workspace.test_schema.retail_sales
USING DELTA
AS SELECT * FROM delta.`/Volumes/workspace/test_schema/sales/retail_sales_delta`;

In [0]:
%sql
insert into test_schema.retail_sales_agg as
SELECT
    category,
    SUM(sales_amount) AS total_sales,
    SUM(quantity) AS qty_sold
    FROM test_schema.retail_sales
    where order_id<>'${order_id}'
GROUP BY category;
